# 项目四：自动化代码生成与执行环境 — Code Sandbox（代码沙箱）

在前三个项目中，我们已经学会了：
- 项目一：用 Prompt 控制大模型输出
- 项目二：让大模型调用外部工具（Tool Use）
- 项目三：让大模型记住对话上下文（Memory）

现在我们要解锁一个更强大的能力——**让大模型自己写代码，自己执行**。

> 这就是 OpenAI 的 **Code Interpreter**（代码解释器）功能的核心原理。

本项目将学习：

1. 理解 **Code Sandbox** 的核心思想：让模型生成代码 → 系统执行 → 结果返回模型
2. 掌握 Python 的 `exec()` 和 `io` 模块来捕获代码输出
3. 学会让模型根据执行结果**自我修正**代码
4. 构建一个能"写代码、跑代码、改代码"的自动化智能体

---

## 一、什么是 Code Sandbox？

### 1.1 从"说话"到"做事"

到目前为止，大模型只能**用文字回答**你。但很多事情用代码做更靠谱：

| 任务 | 让模型直接回答 | 让模型写代码执行 |
|------|---------------|------------------|
| 计算 1234 × 5678 | 可能算错 | 100% 准确 |
| 分析 CSV 数据 | 无法处理 | 写 pandas 代码分析 |
| 画一个图表 | 做不到 | 写 matplotlib 代码 |
| 批量处理文件 | 做不到 | 写循环代码处理 |

**Code Sandbox 的本质：让大模型变成程序员，写代码 → 执行 → 看结果 → 有问题就改。**

### 1.2 核心流程

```
┌──────────────────────────────────────────────────────────┐
│  用户提出任务                                            │
│    ↓                                                     │
│  大模型编写 Python 代码                                   │
│    ↓                                                     │
│  系统在沙箱中执行代码，捕获输出                            │
│    ↓                                                     │
│  把执行结果返回给大模型                                    │
│    ↓                                                     │
│  大模型根据结果判断：                                      │
│    ├── 成功 → 把结果整理成自然语言，返回给用户              │
│    └── 失败 → 分析错误，修改代码，重新执行（最多 N 次）     │
└──────────────────────────────────────────────────────────┘
```

### 1.3 安全警告

```
⚠️ 重要提醒：
本项目使用 exec() 执行代码，仅用于学习演示！
在生产环境中，必须使用隔离的沙箱环境（如 Docker 容器），
防止恶意代码删除文件、访问网络等危险操作。
```

---

## 二、基础准备

### 2.1 导入依赖

本项目需要用到 `io` 模块来捕获代码的 `print()` 输出，以及 `openai` 来调用 DeepSeek API：

In [ ]:
# Ensure openai is installed in JupyterLite/Pyodide before importing it.
try:
    from openai import OpenAI
except ModuleNotFoundError:
    import micropip
    await micropip.install("openai")
    from openai import OpenAI

import os
import io
import sys
import re

# 初始化 DeepSeek 客户端
api_key = os.environ.get("DEEPSEEK_API_KEY", "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

def call_llm(messages, temperature=0.7):
    """调用 DeepSeek API"""
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

print("环境准备完成！")

### 2.2 Python 代码执行基础：exec() 与输出捕获

在让大模型写代码之前，我们先搞清楚**如何在 Python 中执行一段字符串代码**。

#### exec() — 执行字符串代码

```python
code = "print('Hello, World!')"
exec(code)  # 输出: Hello, World!
```

`exec()` 可以执行任意 Python 代码字符串。但它有一个问题：**代码的 print 输出会直接打印到控制台，我们无法"捕获"它。**

#### 如何捕获 print 输出？

我们需要用 `io.StringIO` 临时替换 `sys.stdout`（标准输出流）：

```python
import io
import sys

code = "print('结果是:', 42)"

# 第 1 步：创建一个"虚拟输出流"
output_buffer = io.StringIO()

# 第 2 步：把标准输出重定向到虚拟流
old_stdout = sys.stdout
sys.stdout = output_buffer

try:
    # 第 3 步：执行代码
    exec(code)
finally:
    # 第 4 步：恢复标准输出（必须做！否则后续 print 都看不到）
    sys.stdout = old_stdout

# 第 5 步：从虚拟流中取出输出
captured_output = output_buffer.getvalue()
print(f"捕获到的输出: {captured_output}")  # 捕获到的输出: 结果是: 42
```

#### 封装成工具函数

我们把上面的逻辑封装成一个可复用的函数：

In [ ]:
def run_code(code_string):
    """
    在本地执行一段 Python 代码，并捕获其输出。
    
    参数:
        code_string: 要执行的 Python 代码字符串
    
    返回:
        dict: {"success": True/False, "output": "输出内容", "error": "错误信息"}
    """
    output_buffer = io.StringIO()
    old_stdout = sys.stdout
    old_stderr = sys.stderr
    sys.stdout = output_buffer
    sys.stderr = output_buffer
    
    try:
        exec(code_string)
        sys.stdout = old_stdout
        sys.stderr = old_stderr
        return {
            "success": True,
            "output": output_buffer.getvalue().strip(),
            "error": None
        }
    except Exception as e:
        sys.stdout = old_stdout
        sys.stderr = old_stderr
        return {
            "success": False,
            "output": output_buffer.getvalue().strip(),
            "error": str(e)
        }


# ===== 测试 run_code =====
print("--- 测试 1：正常代码 ---")
result = run_code("print('1 + 1 =', 1 + 1)")
print(f"成功: {result['success']}, 输出: {result['output']}")

print("\n--- 测试 2：多行代码 ---")
result = run_code("""
for i in range(3):
    print(f"第 {i+1} 次循环")
""")
print(f"成功: {result['success']}, 输出: {result['output']}")

print("\n--- 测试 3：会报错的代码 ---")
result = run_code("print(1 / 0)")
print(f"成功: {result['success']}, 错误: {result['error']}")

### 2.3 从模型回复中提取代码

大模型返回的回复通常是**自然语言 + 代码块**混合的，我们需要从中提取出纯代码：

```markdown
好的，我来帮你计算：

```python
result = 1234 * 5678
print(f"结果是: {result}")
```

结果是 7006652。
```

我们用正则表达式提取 ` ```python ... ``` ` 之间的代码：

In [ ]:
def extract_code(response_text):
    """
    从大模型的回复中提取 Python 代码块。
    
    支持的格式：
        ```python
        print("hello")
        ```
    或
        ```
        print("hello")
        ```
    """
    # 匹配 ```python ... ``` 或 ``` ... ```
    pattern = r'```(?:python)?\s*\n(.*?)```'
    matches = re.findall(pattern, response_text, re.DOTALL)
    
    if matches:
        # 返回第一个代码块
        return matches[0].strip()
    return None


# ===== 测试 =====
sample_response = """好的，我来帮你计算：

```python
result = 1234 * 5678
print(f"结果是: {result}")
```

计算完成！"""

code = extract_code(sample_response)
print(f"提取到的代码:\n{code}")

## 三、示例一：数据分析助手

### 3.1 场景

用户给一组销售数据，让智能体自动写代码分析趋势、计算统计值。

### 3.2 代码实现

In [ ]:
def data_analysis_agent(task, data_context=""):
    """
    数据分析智能体：接收任务描述，自动生成代码、执行、返回结果。
    """
    system_prompt = """你是一个数据分析专家。用户会给你一段数据和任务，你需要：
1. 编写 Python 代码来完成任务
2. 代码必须用 print() 输出结果
3. 只返回代码块（用 ```python 包裹），不要多余的解释
4. 不要使用 matplotlib 等画图库，只用纯文本输出
5. 不要使用 input() 函数"""
    
    user_prompt = f"任务：{task}"
    if data_context:
        user_prompt += f"\n\n数据：\n{data_context}"
    
    # 第 1 步：让模型生成代码
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    response = call_llm(messages, temperature=0.3)
    
    print("=== 模型生成的代码 ===")
    code = extract_code(response)
    if code is None:
        print("（模型没有返回代码块）")
        return response
    print(code)
    
    # 第 2 步：执行代码
    print("\n=== 代码执行结果 ===")
    result = run_code(code)
    
    if result["success"]:
        print(result["output"])
        
        # 第 3 步：把执行结果返回给模型，让它做总结
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user", "content": f"代码执行成功，输出如下：\n{result['output']}\n\n请用自然语言总结分析结果。"})
        summary = call_llm(messages, temperature=0.5)
        
        print("\n=== 智能体总结 ===")
        print(summary)
        return summary
    else:
        print(f"代码执行出错: {result['error']}")
        return f"代码执行出错: {result['error']}"


# ===== 测试 =====
sales_data = """
月份, 销售额(万元)
1月, 12.5
2月, 10.8
3月, 15.2
4月, 18.6
5月, 22.1
6月, 19.8
"""

data_analysis_agent(
    task="请分析销售数据的趋势，计算平均值、最高值、最低值，并判断整体是上升还是下降。",
    data_context=sales_data
)

### 3.3 运行分析

运行上面的代码，你会看到完整的流程：

1. **模型生成代码** — 用 pandas 或纯 Python 分析数据
2. **代码执行** — `run_code()` 执行代码，捕获 print 输出
3. **模型总结** — 把执行结果返回给模型，模型用自然语言总结

这就是 Code Interpreter 的核心：**模型写代码 → 系统执行 → 结果回传模型 → 模型总结**。

---

## 四、示例二：带自我修正的代码智能体

### 4.1 为什么需要自我修正？

模型生成的代码**不一定一次就跑对**。常见错误：
- 变量名写错
- 逻辑 bug
- 使用了不存在的库

所以我们需要一个**自动纠错循环**：

```
生成代码 → 执行 → 出错？
                ├── 是 → 把错误信息返回给模型，让它修改代码 → 重新执行
                └── 否 → 返回结果
```

### 4.2 代码实现

In [ ]:
def self_fixing_agent(task, max_retries=3):
    """
    自我修正智能体：如果代码执行出错，自动把错误信息返回给模型，让它修改代码。
    最多重试 max_retries 次。
    """
    system_prompt = """你是一个 Python 编程专家。用户会给你一个任务，你需要：
1. 编写 Python 代码来完成任务
2. 代码必须用 print() 输出结果
3. 只返回代码块（用 ```python 包裹），不要多余的解释
4. 不要使用 matplotlib 等画图库，只用纯文本输出
5. 不要使用 input() 函数
6. 不要使用网络请求"""
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"请完成以下任务：{task}"}
    ]
    
    for attempt in range(1, max_retries + 1):
        print(f"\n{'='*50}")
        print(f"第 {attempt} 次尝试")
        print(f"{'='*50}")
        
        # 第 1 步：让模型生成/修改代码
        response = call_llm(messages, temperature=0.3)
        
        code = extract_code(response)
        if code is None:
            print("模型没有返回代码块，回复内容：")
            print(response)
            return response
        
        print("--- 生成的代码 ---")
        print(code)
        
        # 第 2 步：执行代码
        result = run_code(code)
        
        if result["success"]:
            print(f"\n--- 执行成功 ---")
            print(f"输出: {result['output']}")
            
            # 让模型总结结果
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": f"代码执行成功，输出：\n{result['output']}\n\n请用自然语言总结结果。"})
            summary = call_llm(messages, temperature=0.5)
            print(f"\n--- 智能体总结 ---")
            print(summary)
            return summary
        else:
            print(f"\n--- 执行出错 ---")
            print(f"错误: {result['error']}")
            
            # 把错误信息返回给模型，让它修改
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": f"代码执行出错了：\n错误信息: {result['error']}\n\n请分析错误原因，修改代码后重新提交。"})
            print("正在让模型修正代码...")
    
    print(f"\n已达到最大重试次数 ({max_retries})，任务失败。")
    return "任务失败：多次尝试后代码仍然出错。"


# ===== 测试 1：简单任务（通常一次成功）=====
print("=" * 60)
print("测试 1：生成斐波那契数列")
print("=" * 60)
self_fixing_agent("请生成前 10 个斐波那契数，并计算它们的总和。")


# ===== 测试 2：稍微复杂的任务（可能需要修正）=====
print("\n\n" + "=" * 60)
print("测试 2：字符串处理")
print("=" * 60)
self_fixing_agent(
    "给定字符串 'Hello World Python Programming'，"
    "请统计每个单词的长度，找出最长的单词，"
    "并按长度从长到短排序输出。"
)

### 4.3 运行分析

运行上面的代码，观察输出：

1. **测试 1**：通常第 1 次就成功，模型生成斐波那契代码并正确执行
2. **测试 2**：也可能第 1 次成功，但如果出错，你会看到：
   - 错误信息被返回给模型
   - 模型分析错误原因，生成修改后的代码
   - 重新执行，直到成功或达到最大重试次数

**自我修正的关键：** 把错误信息原封不动地返回给模型，模型能"看懂"报错并修复。

**两种示例的对比：**

| | 示例一（数据分析） | 示例二（自我修正） |
|---|---|---|
| 执行次数 | 固定 1 次 | 最多 N 次（自动重试） |
| 出错处理 | 直接返回错误 | 把错误返回给模型修正 |
| 适用场景 | 简单任务 | 复杂任务，容错性更高 |

---

## 五、课后练习

### 练习 1（基础）：数学计算器智能体

**目标：** 构建一个能解决数学问题的智能体。用户输入数学问题，智能体自动生成代码计算并返回结果。

**要求：**
1. 补全 `math_agent` 函数
2. 设计 System Prompt，让模型只返回可执行的 Python 代码（用 ` ```python ` 包裹）
3. 提取代码 → 执行代码 → 把结果返回给用户
4. 不需要自我修正，执行出错时返回错误信息即可

**提示：**
- System Prompt 要明确告诉模型："只返回代码块，不要多余解释"
- 使用 `extract_code()` 提取代码，`run_code()` 执行代码
- 模型可以让代码使用 `math` 模块

**测试用例：**
- "计算 2 的 20 次方"
- "一个圆的半径是 5，请计算它的面积和周长"
- "求 1 到 100 之间所有偶数的和"

In [ ]:
# ===== 练习 1：数学计算器智能体 =====

def math_agent(question):
    """数学计算器智能体"""
    # TODO: 1. 设计 System Prompt
    # 提示：告诉模型它是一个数学计算专家，需要编写 Python 代码来解题
    # 要求模型只返回 ```python 代码块，用 print() 输出结果
    system_prompt = ""
    
    # TODO: 2. 调用模型生成代码
    # messages = [...]
    # response = call_llm(messages, temperature=0.3)
    
    # TODO: 3. 提取代码
    # code = extract_code(response)
    
    # TODO: 4. 执行代码
    # result = run_code(code)
    
    # TODO: 5. 返回结果（成功则返回输出，失败则返回错误信息）
    pass


# ===== 测试 =====
print("=== 测试 1 ===")
print(math_agent("计算 2 的 20 次方"))

print("\n=== 测试 2 ===")
print(math_agent("一个圆的半径是 5，请计算它的面积和周长"))

print("\n=== 测试 3 ===")
print(math_agent("求 1 到 100 之间所有偶数的和"))

### 练习 2（进阶）：带自我修正的数据处理智能体

**目标：** 构建一个能处理 JSON 数据的智能体，具备**自我修正**能力。

**场景：** 用户给一段 JSON 数据和一个处理任务，智能体自动写代码处理。如果代码出错，自动修正并重试。

**要求：**
1. 补全 `data_processor_agent` 函数
2. 实现**最多 3 次重试**的自我修正循环
3. 每次出错时，把错误信息和之前的代码一起返回给模型修正
4. 成功后用自然语言总结结果

**提示：**
- 参考示例二的 `self_fixing_agent` 实现重试循环
- 在 System Prompt 中告诉模型数据格式是 JSON
- 模型可以让代码使用 `json` 模块
- 注意：每次重试时，messages 列表要包含之前的代码和错误信息

**测试用例：**
- 任务 1："找出成绩最高的学生，并计算全班平均分"
- 任务 2："按成绩从高到低排序，并统计及格人数（>=60分）"
- 任务 3："找出名字中包含'张'的学生，并列出他们的成绩"

In [ ]:
import json

# ===== 练习 2：带自我修正的数据处理智能体 =====

# 测试数据
student_data = json.dumps([
    {"name": "张三", "score": 85, "grade": "高三"},
    {"name": "李四", "score": 92, "grade": "高二"},
    {"name": "王五", "score": 58, "grade": "高三"},
    {"name": "张小明", "score": 76, "grade": "高一"},
    {"name": "赵六", "score": 43, "grade": "高二"},
    {"name": "张伟", "score": 67, "grade": "高三"},
], ensure_ascii=False)


def data_processor_agent(task, json_data, max_retries=3):
    """带自我修正的数据处理智能体"""
    # TODO: 1. 设计 System Prompt
    # 提示：告诉模型它是一个数据处理专家，会收到 JSON 数据和处理任务
    # 要求模型编写 Python 代码处理数据，用 print() 输出结果
    system_prompt = ""
    
    # TODO: 2. 构建初始 messages
    # 提示：user 消息应包含任务和 JSON 数据
    messages = []
    
    # TODO: 3. 实现重试循环
    # for attempt in range(1, max_retries + 1):
    #     3a. 调用模型生成代码
    #     3b. 提取代码
    #     3c. 执行代码
    #     3d. 如果成功：让模型总结结果，返回
    #     3e. 如果失败：把错误信息加入 messages，继续循环
    pass


# ===== 测试 =====
print("=== 测试 1：找出最高分和平均分 ===")
data_processor_agent(
    task="找出成绩最高的学生，并计算全班平均分",
    json_data=student_data
)

print("\n=== 测试 2：排序和统计 ===")
data_processor_agent(
    task="按成绩从高到低排序，并统计及格人数（>=60分）",
    json_data=student_data
)

print("\n=== 测试 3：条件筛选 ===")
data_processor_agent(
    task="找出名字中包含'张'的学生，并列出他们的成绩",
    json_data=student_data
)

---
## 小结

恭喜你完成了第四个项目！回顾一下核心知识点：

| 知识点 | 要点 |
|--------|------|
| Code Sandbox 的本质 | 模型写代码 → 系统执行 → 结果回传模型 |
| `exec()` | 执行字符串形式的 Python 代码 |
| `io.StringIO` | 捕获代码的 print 输出 |
| 代码提取 | 用正则表达式从模型回复中提取 ` ```python ` 代码块 |
| 自我修正 | 把错误信息返回给模型，让它修改代码重新执行 |
| 安全注意 | 生产环境必须使用隔离沙箱（如 Docker），不能直接用 exec() |

**核心流程：** 任务 → 模型写代码 → 执行 → 成功则总结 / 失败则修正 → 循环

**下一步预告：** 在[项目五](./project5_multi_agent.ipynb)中，我们将学习**多智能体协作**——让多个 AI 角色分工合作，完成更复杂的任务！